In [1]:
import math
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torchvision
torchvision.disable_beta_transforms_warning()
from scipy.special import expit
from sklearn.metrics import roc_auc_score

import sys
sys.path.insert(0, str(Path.cwd().parent))  # repo root, for experiment_config / dataloader / models
from experiment_config import config
from dataloader import get_data_loader

data_dir = "/datasets/slow/LUNA25"
workspace_root = "/models/hiera-luna25"
results_dir = f"{workspace_root}/results/finetune-hiera-3D-20260814"
out_dir = Path(f"{workspace_root}/figures/diagnostics")
S = [0.67, 0.8, 0.9, 1.0, 1.1, 1.25, 1.5]
df_pred = pd.read_csv(f"{results_dir}/all_folds_predictions.csv")
csv_path = out_dir / "rescale_predictions.csv"
if not csv_path.exists():
    raise FileNotFoundError(f"{csv_path} missing: run `cd jobs && sbatch run_analysis.sh analysis/diagnostics_rescale.py` first")
res = pd.read_csv(csv_path)
print(res.shape, res["scale"].unique())

(34776, 5) [0.67 0.8  0.9  1.   1.1  1.25 1.5 ]


In [2]:
# (a) sanity check at s=1
r1 = res[res["scale"] == 1.0].merge(df_pred[["AnnotationID", "probability"]], on="AnnotationID")
chk = r1.groupby("fold").apply(lambda g: pd.Series({"max_abs_diff": (expit(g["logit"]) - g["probability"]).abs().max(),
                                                      "auc": roc_auc_score(g["label"], g["logit"])}))
display(chk)

# (b) per-scale table
piv = res.pivot_table(index=["AnnotationID", "label"], columns="scale", values="logit").reset_index(level="label")
d = piv[S].sub(piv[1.0], axis=0)
rows = {}
for s in S:
    sub = res[res["scale"] == s]
    rows[s] = {"pooled_auc": roc_auc_score(sub["label"], sub["logit"]),
               "mean_fold_auc": np.mean([roc_auc_score(g["label"], g["logit"]) for _, g in sub.groupby("fold")]),
               "mean_dlogit_l0": d.loc[piv["label"] == 0, s].mean(), "mean_dlogit_l1": d.loc[piv["label"] == 1, s].mean(),
               "median_dlogit": d[s].median()}
display(pd.DataFrame(rows).T.round(4))

# (c) per-nodule slope of logit on log(s)
ls = np.log(S); lc = ls - ls.mean()
slope = pd.Series((piv[S].values - piv[S].values.mean(1, keepdims=True)) @ lc / (lc @ lc), index=piv.index).rename("slope")
sl = pd.DataFrame({"slope": slope, "label": piv["label"]})
def summ(x): return pd.Series({"n": len(x), "median": x.median(), "q25": x.quantile(.25), "q75": x.quantile(.75), "frac_pos": (x > 0).mean()})
display(pd.DataFrame({"all": summ(sl["slope"]), "label0": summ(sl.loc[sl.label == 0, "slope"]), "label1": summ(sl.loc[sl.label == 1, "slope"])}).T.round(4))

# (d) cross-sectional slope: s=1 logit ~ log(diameter)
ann = pd.read_csv(f"{data_dir}/LUNA25_Public_Training_Development_Data.csv")
vol = pd.read_csv(f"{workspace_root}/mask_volumes.csv")
vol["SeriesInstanceUID"] = vol["filename"].str.replace(".nii.gz", "", regex=False)
dm = ann.merge(vol.drop(columns="filename"), on="SeriesInstanceUID", how="left")[["AnnotationID", "volume_mm3"]]
dm["diameter_mm"] = 2 * ((3 * dm["volume_mm3"]) / (4 * math.pi)) ** (1 / 3)
x1 = res[res["scale"] == 1.0].merge(dm, on="AnnotationID").dropna(subset=["diameter_mm"])
x1 = x1[x1["diameter_mm"] > 0]
def ols(x, y): return np.polyfit(x, y, 1)[0]
print(f"cross-sectional slope (n={len(x1)}): {ols(np.log(x1['diameter_mm']), x1['logit']):.4f}")

# (e) per diameter bin
sl2 = sl.reset_index().merge(dm, on="AnnotationID")
sl2["bin"] = pd.cut(sl2["diameter_mm"], [0, 6, 8, 15, np.inf]); x1["bin"] = pd.cut(x1["diameter_mm"], [0, 6, 8, 15, np.inf])
display(pd.DataFrame({"n": x1.groupby("bin", observed=False).size(),
                      "counterfactual_median_slope": sl2.groupby("bin", observed=False)["slope"].median(),
                      "cross_sectional_slope": x1.groupby("bin", observed=False).apply(lambda g: ols(np.log(g["diameter_mm"]), g["logit"]))}).round(4))

,max_abs_diff,auc
fold,,
1,8.968843e-08,0.902475
2,1.063548e-07,0.890164
3,9.685290e-08,0.912684
4,9.002808e-08,0.887039
5,9.803166e-08,0.900321


,pooled_auc,mean_fold_auc,mean_dlogit_l0,mean_dlogit_l1,median_dlogit
0.67,0.8548,0.8561,-0.3508,-0.6860,-0.2256
0.80,0.8781,0.8808,-0.2418,-0.3577,-0.1397
0.90,0.8908,0.8937,-0.1199,-0.1450,-0.0575
1.00,0.8952,0.8985,0.0000,0.0000,0.0000
1.10,0.8917,0.8962,0.1189,0.0752,0.0570
1.25,0.8846,0.8897,0.3481,0.1968,0.2391
1.50,0.8630,0.8686,0.7711,0.3389,0.6301


,n,median,q25,q75,frac_pos
all,4968.0,1.2757,0.4569,2.1805,0.8931
label0,4510.0,1.3000,0.4680,2.1897,0.8951
label1,458.0,1.0969,0.3895,2.0044,0.8734


cross-sectional slope (n=4955): 1.0891


,n,counterfactual_median_slope,cross_sectional_slope
bin,,,
"(0.0, 6.0]",1065,1.1185,0.8306
"(6.0, 8.0]",1193,1.4151,0.5653
"(8.0, 15.0]",1958,1.3784,1.2058
"(15.0, inf]",739,1.0543,0.9555


In [3]:
# (f) mean delta logit vs s
fig, ax = plt.subplots(figsize=(6, 4))
for l in (0, 1):
    dl = d[piv["label"] == l]
    ax.plot(S, dl.mean(), marker="o", label=f"label {l} (mean)")
    ax.fill_between(S, dl.quantile(.25), dl.quantile(.75), alpha=.2)
ax.set_xscale("log"); ax.set_xticks(S); ax.set_xticklabels(S); ax.minorticks_off()
ax.set_xlabel("scale s"); ax.set_ylabel("delta logit vs s=1"); ax.legend(); fig.tight_layout()
fig.savefig(out_dir / "rescale_delta_logit.png", dpi=150)

# (g) example crops, middle depth slice
ids = res[(res["scale"] == 1.0)].sort_values("AnnotationID")["AnnotationID"].iloc[[0, 1500, 3000]]
fig, axs = plt.subplots(3, 3, figsize=(9, 9))
for i, aid in enumerate(ids):
    for j, s in enumerate([0.67, 1.0, 1.5]):
        b = next(iter(get_data_loader("/datasets/slow/LUNA25/luna25_nodule_blocks", df_pred[df_pred["AnnotationID"] == aid], mode="3D",
                                      size_px=config.SIZE_PX, size_mm=config.SIZE_MM / s)))
        axs[i, j].imshow(b["image"][0, 0, config.DEPTH_PX // 2], cmap="gray"); axs[i, j].axis("off"); axs[i, j].set_title(f"{aid} s={s}", fontsize=8)
fig.tight_layout(); fig.savefig(out_dir / "rescale_examples.png", dpi=150)